# Class 3 - Building a Simple Agent

**Week 5: Introduction to AI Agents**

### Learning objectives
By the end of this notebook you will be able to:
- Explain why short-term memory matters across turns.
- Build a multi-tool LangChain agent on Groq (calculator + mock search).
- Add guardrails: recursion limits and safe tool error strings.
- Know when to look at **LangGraph** (complex graphs) or **LlamaIndex** (another agent stack) after this course pattern.

Week 6 will cover retrieval / vector memory — keep memory here to chat history + optional scratch notes.

## Setup

In [5]:
!pip install -q langchain langchain-groq langchain_community langgraph

In [12]:
import os
import re
import uuid

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    print("No GROQ_API_KEY found. Live agent cells will skip.")
else:
    print("Found GROQ_API_KEY. Ready to build the agent.")

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver


def make_llm():
    if not GROQ_API_KEY:
        return None
    return ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)


def build_memory_agent(tools, system_prompt=None, max_turns=8):
    llm = make_llm()
    if llm is None:
        return None
    prompt = system_prompt or (
        "You are a helpful multi-tool assistant. "
        "Use calculator for math and mock_search for facts in the catalog. "
        "If a tool errors, explain the problem and ask for a corrected input. "
        "You cannot store new information or remember things; you can only look up facts you already know with mock_search." # Added this line
    )
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=prompt,
        checkpointer=InMemorySaver(),
    )
    agent._max_turns = max_turns
    return agent


def chat(agent, text: str, thread_id: str = "week5-class3"):
    if agent is None:
        print("Skipping — no agent / key.")
        return None
    result = agent.invoke(
        {"messages": ([{"role": "user", "content": text}])},
        config={
            "configurable": {"thread_id": thread_id},
            "recursion_limit": getattr(agent, "_max_turns", 8),
        },
    )
    final = result["messages"][-1]
    content = getattr(final, "content", final)
    print(content)
    return result


Found GROQ_API_KEY. Ready to build the agent.


## 1. Why memory matters

Run the cell below twice in spirit:
1. **Without memory** — remember Kathmandu on one `thread_id`, then ask the follow-up on a **new** `thread_id` (no shared checkpointer history).
2. **With memory** — run both turns on the **same** `thread_id` so `InMemorySaver` keeps the earlier message.

Same follow-up question; only the thread changes. If `GROQ_API_KEY` is missing, the cell prints a skip message.


In [13]:
from langchain.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '12 * 1.8 + 5'."""
    expr = expression.strip()
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expr):
        return "Calculator error: only digits and + - * / ( ) are allowed."
    try:
        value = eval(expr, {"__builtins__": {}}, {})
    except Exception as e:
        return f"Calculator error: {e}"
    return str(value)


@tool
def mock_search(query: str) -> str:
    """Look up a short fact from a tiny local catalog (not the real web)."""
    catalog = {
        "kathmandu elevation": "Kathmandu sits about 1,400 meters above sea level.",
        "nepal capital": "Kathmandu is the capital of Nepal.",
        "water boil celsius": "Pure water boils at 100°C at standard pressure.",
    }
    q = query.strip().lower()
    for key, value in catalog.items():
        if key in q or q in key:
            return value
    return "No catalog hit. Try queries like 'Nepal capital' or 'Kathmandu elevation'."


demo_tools = [calculator, mock_search]
agent = build_memory_agent(demo_tools)

remember_msg = "Remember that my destination city is Kathmandu."
follow_up = "Using mock_search, what is the elevation there?"

if agent is None:
    print("Skipping memory contrast — no agent / GROQ_API_KEY.")
else:
    print("=== Without shared history (new thread_id on the follow-up) ===")
    chat(agent, remember_msg, thread_id=f"no-memory-{uuid.uuid4()}")
    chat(agent, follow_up, thread_id=f"no-memory-{uuid.uuid4()}")

    print("\n=== With memory (same thread_id for both turns) ===")
    shared_thread = f"memory-demo-{uuid.uuid4()}"
    chat(agent, remember_msg, thread_id=shared_thread)
    chat(agent, follow_up, thread_id=shared_thread)


=== Without shared history (new thread_id on the follow-up) ===
I can't remember things or store new information. If you'd like to know more about Kathmandu, I can try looking up facts in the catalog. What would you like to know?
To find the elevation of a specific location, I need to know the location. Please provide a location, such as a city or mountain, so I can look up its elevation using mock_search.

=== With memory (same thread_id for both turns) ===
I can't remember that your destination city is Kathmandu. I'm a helpful multi-tool assistant, but I don't have the ability to store new information or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to know something about Kathmandu, I can try to look it up for you with the mock_search function.
The elevation of Kathmandu is about 1,400 meters above sea level.


## 2. Two kinds of memory (foundations level)

- **Short-term:** the message list for a `thread_id` (what `InMemorySaver` keeps for this process).
- **Scratch notes:** a string or dict *you* maintain and inject into the next user message when needed.

Vector databases / RAG document memory arrive in Week 6 — do not add them here.

In [14]:
NOTES = {"destination": None}

def note_aware_chat(agent, text: str, thread_id: str = "notes-demo"):
    # Lightweight scratchpad: prepend known notes so the model can use them
    preface = ""
    if NOTES.get("destination"):
        preface = f"(Instructor notes: destination={NOTES['destination']})\n"
    return chat(agent, preface + text, thread_id=thread_id)

NOTES["destination"] = "Pokhara"
note_aware_chat(agent, "Should I search for the capital or stay focused on my destination?")

Since Kathmandu is the capital of Nepal, and your destination is Pokhara, you may want to stay focused on Pokhara as it is not the capital.


{'messages': [HumanMessage(content='(Instructor notes: destination=Pokhara)\nShould I search for the capital or stay focused on my destination?', additional_kwargs={}, response_metadata={}, id='00454bf2-f132-4240-8762-e5b154aa06c7'),
  AIMessage(content="You should stay focused on your destination, which is Pokhara. If you need to know more about Pokhara, I can help you with that. However, if you're looking for information on the capital of Nepal, I can do that as well. \n\n", additional_kwargs={'tool_calls': [{'id': 'ab454v6jp', 'function': {'arguments': '{"query":"What is the capital of Nepal"}', 'name': 'mock_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 384, 'total_tokens': 458, 'completion_time': 0.183636695, 'completion_tokens_details': None, 'prompt_time': 0.02053444, 'prompt_tokens_details': None, 'queue_time': 0.007948305, 'total_time': 0.204171135}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': '

## 3. Guardrails

- Cap iterations with `recursion_limit` / `max_turns`.
- Return tool errors as strings (calculator already does).
- Keep the tool list explicit — only register tools you intend to allow.

In [15]:
strict_agent = build_memory_agent(demo_tools, max_turns=5)
chat(strict_agent, "What is 17 * 19? Then remind me of Nepal's capital.", thread_id="guard-demo")

The result of 17 * 19 is 323. The capital of Nepal is Kathmandu.


{'messages': [HumanMessage(content="What is 17 * 19? Then remind me of Nepal's capital.", additional_kwargs={}, response_metadata={}, id='8204019e-8330-4b66-ba84-ef2f6a609469'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6j65pz0a1', 'function': {'arguments': '{"expression":"17 * 19"}', 'name': 'calculator'}, 'type': 'function'}, {'id': 's4j67e7nb', 'function': {'arguments': '{"query":"Nepal capital"}', 'name': 'mock_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 377, 'total_tokens': 411, 'completion_time': 0.073072233, 'completion_tokens_details': None, 'prompt_time': 0.020813045, 'prompt_tokens_details': None, 'queue_time': 0.008536955, 'total_time': 0.093885278}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fefb7-98b6-7950-8d1c-8b81a7888150-0', to

## Closing

You shipped a small agent: tools, memory, and limits. For branching multi-actor workflows, explore **LangGraph** next. If your team standardizes on **LlamaIndex**, the same ideas transfer — tools, a loop, memory, and stop conditions.

**Next week:** Foundations of RAG & Chatbots.

## Challenges

### Challenge 01 — `unit_convert`
Add `@tool def unit_convert(value: float, from_unit: str, to_unit: str) -> str` supporting at least `celsius↔fahrenheit` and `km↔miles`. Register it and test.

In [ ]:
# TODO
pass

In [21]:
@tool
def unit_convert(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between Celsius/Fahrenheit and kilometers/miles."""

    from_unit = from_unit.lower().strip()
    to_unit = to_unit.lower().strip()

    if from_unit == "celsius" and to_unit == "fahrenheit":
        result = (value * 9/5) + 32
    elif from_unit == "fahrenheit" and to_unit == "celsius":
        result = (value - 32) * 5/9
    elif from_unit == "km" and to_unit == "miles":
        result = value * 0.621371
    elif from_unit == "miles" and to_unit == "km":
        result = value * 1.60934
    else:
        return "Unsupported conversion. Use celsius↔fahrenheit or km↔miles."

    return f"{value} {from_unit} = {result:.2f} {to_unit}"

In [23]:
print(unit_convert.invoke({
    "value": 100,
    "from_unit": "celsius",
    "to_unit": "fahrenheit"
}))

print(unit_convert.invoke({
    "value": 10,
    "from_unit": "km",
    "to_unit": "miles"
}))

100.0 celsius = 212.00 fahrenheit
10.0 km = 6.21 miles


### Challenge 02 — Expand `mock_search`
Add three new catalog entries and demonstrate a successful lookup for one of them.

In [ ]:
# TODO
pass

In [20]:
# Add three new entries to the catalog
catalog = {
    "kathmandu elevation": "Kathmandu sits about 1,400 meters above sea level.",
    "nepal capital": "Kathmandu is the capital of Nepal.",
    "water boil celsius": "Pure water boils at 100°C at standard pressure.",

    "nepal currency": "The currency of Nepal is the Nepalese rupee.",
    "mount everest height": "Mount Everest is about 8,849 meters tall.",
    "nepal largest city": "Kathmandu is the largest city in Nepal.",
}

# Demonstrate a successful lookup
query = "What is the currency of Nepal?"
q = query.strip().lower()

for key, value in catalog.items():
    if key in q or q in key:
        print(value)
        break
else:
    print("No catalog hit.")

No catalog hit.


### Challenge 03 — Harden the calculator
Extend validation (e.g. reject `**`, `//`, or empty input) and show the agent recovering from a bad expression via the error string.

In [ ]:
# TODO
pass

In [22]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a safe basic arithmetic expression."""

    expr = expression.strip()

    # Reject empty input
    if not expr:
        return "Calculator error: expression cannot be empty."

    # Reject unsupported operators
    if "**" in expr:
        return "Calculator error: exponentiation (**) is not allowed."

    if "//" in expr:
        return "Calculator error: floor division (//) is not allowed."

    # Only allow digits, operators, parentheses, spaces, and decimal points
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expr):
        return "Calculator error: only digits and + - * / ( ) . are allowed."

    try:
        value = eval(expr, {"__builtins__": {}}, {})
    except Exception as e:
        return f"Calculator error: {e}"

    return str(value)

In [24]:
print(calculator.invoke({"expression": ""}))
print(calculator.invoke({"expression": "2 ** 3"}))
print(calculator.invoke({"expression": "10 // 3"}))
print(calculator.invoke({"expression": "10 / 2"}))

Calculator error: expression cannot be empty.
Calculator error: exponentiation (**) is not allowed.
Calculator error: floor division (//) is not allowed.
5.0


### Challenge 04 (optional) — Persistent notes
Update `NOTES` from user text (e.g. parse "my destination is X") and show a later answer that depends on the note.

In [ ]:
# TODO
pass

In [27]:
NOTES = {}

def update_notes(user_text: str):
    text = user_text.strip()

    # Look for: "my destination is X"
    match = re.search(r"my destination is (.+)", text, re.IGNORECASE)

    if match:
        destination = match.group(1).strip().rstrip(".")
        NOTES["destination"] = destination
        return f"Saved your destination as {destination}."

    return "No note was saved."


def answer_from_notes():
    destination = NOTES.get("destination")

    if destination:
        return f"Your destination is {destination}."

    return "I don't have a destination saved yet."

In [28]:
print(update_notes("My destination is Kathmandu."))
print(answer_from_notes())

Saved your destination as Kathmandu.
Your destination is Kathmandu.
